# Lab 3 — Curate your own evaluation dataset

**Companion to course Lesson 3.**

BEIR is a fixed benchmark. It tells you how a retrieval setup
does on *scientific claim verification* or *medical literature
retrieval*. That's useful for comparing approaches in the
abstract. It is *not* useful for answering the question your
team actually has: **does this retrieve well on the kind of
queries our users ask, against the documents we actually
have?** For that you need a domain-specific evaluation set
built on your own corpus.

## What you'll build

A small judgement list (queries + relevance labels) over the
corpus you already ingested, using the standard bootstrap
pattern:

1. **Sample** documents from your corpus.
2. **Draft queries** with an LLM — one realistic query per doc.
3. **Pool** candidates by retrieving top-K for each draft
   query (we'll use `$rankFusion`).
4. **Grade** each `(query, candidate)` pair with an LLM judge
   on a 0–3 scale.
5. **Curate** — you spot-check and edit. The LLM is fast and
   consistent; *you* know what counts as relevant in your
   domain.
6. **Save & re-evaluate** — write the judgements to disk and
   rerun Lab 1's metrics against them.

> **Requires `OPENAI_API_KEY` in `.env`.** The full bootstrap
> on 20 docs ≈ 60 LLM calls — a few cents on `gpt-4o-mini`.

In [1]:
import os, sys
_REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

In [2]:
from dotenv import load_dotenv
load_dotenv(os.path.join(_REPO_ROOT, ".env"))

assert os.environ.get("OPENAI_API_KEY"), \
    "Set OPENAI_API_KEY in .env before running this lab."

import math, json, pathlib, pymongo, voyageai
from openai import OpenAI
from lib import embed_queries

DATASET           = "scifact"
DB_NAME           = "voyage_context_demo"
COLL_NAME         = f"chunks_{DATASET.replace('-', '_')}"
VECTOR_INDEX_NAME = "voyage_vector_index"
TEXT_INDEX_NAME   = "voyage_text_index"
MONGODB_BASE_URL  = "https://ai.mongodb.com/v1"

N_DOCS    = 20            # corpus sample for the bootstrap
TOP_K     = 8             # candidates per query to grade
LLM_MODEL = "gpt-4o-mini"

client = pymongo.MongoClient(os.environ["MONGODB_URI"])
coll   = client[DB_NAME][COLL_NAME]
voyage = voyageai.Client(api_key=os.environ["VOYAGE_API_KEY"],
                         base_url=MONGODB_BASE_URL)
oai    = OpenAI()

## Step 1 — Sample documents from the corpus

Pull `N_DOCS` distinct parent documents from our chunks
collection. Each will seed one draft query.

In [3]:
sample_docs = []
seen = set()
for row in coll.find({}, {"doc_id": 1, "title": 1, "text": 1}).limit(N_DOCS * 4):
    did = row['doc_id']
    if did in seen:
        continue
    seen.add(did)
    sample_docs.append({
        "doc_id": did,
        "title" : row.get('title', ''),
        "text"  : row['text'][:1500],
    })
    if len(sample_docs) >= N_DOCS:
        break

print(f"Sampled {len(sample_docs)} documents:")
for d in sample_docs[:3]:
    print(f"  {d['doc_id']:<12} {d['title'][:55]!r}")

Sampled 20 documents:
  14241418     'NVP-BEZ235, a dual PI3K/mTOR inhibitor, prevents PI3K s'
  3863543      'Mesenchymal Inflammation Drives Genotoxic Stress in Hem'
  4406819      'PAAR-repeat proteins sharpen and diversify the Type VI '


## Step 2 — Draft a query for each document

We ask the LLM to write **one realistic query** that a domain
user might type to find each document. The prompt is the
lever — phrase it for your real users (keyword-style vs
natural-language vs multi-hop).

In [4]:
QUERY_PROMPT = (
    "You are helping build a retrieval evaluation set. Given a document, "
    "write ONE concise, realistic search query that a domain user might "
    "type to retrieve this exact document. The query should be 3-15 words "
    "and read naturally — not a paraphrase of the title.\n\n"
    "Document title : {title}\n"
    "Document text  : {text}\n\n"
    "Output ONLY the query, no quotes or commentary."
)

draft_queries = []
for i, d in enumerate(sample_docs, 1):
    resp = oai.chat.completions.create(
        model       = LLM_MODEL,
        temperature = 0.7,
        messages    = [{"role": "user",
                        "content": QUERY_PROMPT.format(title=d['title'], text=d['text'])}],
    )
    q = resp.choices[0].message.content.strip().strip("\"'")
    draft_queries.append({
        "qid"        : f"my_q_{i:03d}",
        "query"      : q,
        "seed_doc_id": d['doc_id'],
    })
    print(f"  [{i:>2}] {q}")

  [ 1] NVP-BEZ235 effects on cancer cells with PI3K mutations


  [ 2] how mesenchymal inflammation affects hematopoietic stem cells in pre-leukemia


  [ 3] How do PAAR-repeat proteins affect the Type VI secretion system?


  [ 4] mTOR signaling and myeloid-derived suppressor cells in tumors


  [ 5] FOXO3 pathway impact on inflammatory and infectious diseases


...

  [16] effects of folic acid on endothelial function in coronary artery disease


  [17] relationship between sun exposure and multiple sclerosis risk


  [18] long term effects of glucose tolerance on blood pressure in men


  [19] microtubule acetylation LRRK2 mutations axonal transport deficits


  [20] how to track adipogenesis in white adipose tissue


## Step 3 — Pool candidates

For each draft query, retrieve the top-K with a hybrid
`$rankFusion`. The union of these results is the **pool** —
the set of `(query, doc)` pairs we'll grade. Pooling is the
standard IR evaluation pattern (TREC + BEIR both use it): we
don't grade every doc in the corpus, only the docs that
*some* retriever surfaced.

In [5]:
# Embed the draft queries in one batch.
q_vecs = embed_queries(voyage, [q['query'] for q in draft_queries])

def hybrid_search(q_vec, q_text, alpha=0.8, top_k=TOP_K):
    EPS = 1e-3
    w_vec, w_text, n = max(alpha, EPS), max(1.0 - alpha, EPS), 50
    pipeline = [
        {"$rankFusion": {
            "input": {"pipelines": {
                "vector": [{"$vectorSearch": {
                    "index": VECTOR_INDEX_NAME, "path": "embedding",
                    "queryVector": q_vec,
                    "numCandidates": n * 4, "limit": n,
                }}],
                "text": [{"$search": {
                    "index": TEXT_INDEX_NAME,
                    "text" : {"path": "text", "query": q_text},
                }}, {"$limit": n}],
            }},
            "combination": {"weights": {"vector": w_vec, "text": w_text}},
        }},
        {"$addFields": {"score": {"$meta": "score"}}},
        {"$limit": top_k * 4},
    ]
    seen = {}
    for row in coll.aggregate(pipeline):
        did = row['doc_id']
        if did not in seen or row['score'] > seen[did]['score']:
            seen[did] = row
    return sorted(seen.values(), key=lambda r: r['score'], reverse=True)[:top_k]

pool = {}
for q, qv in zip(draft_queries, q_vecs):
    pool[q['qid']] = hybrid_search(qv, q['query'])

total = sum(len(v) for v in pool.values())
print(f"Pooled {total} (query, candidate) pairs across {len(pool)} queries.")

Pooled 160 (query, candidate) pairs across 20 queries.


## Step 4 — LLM judge

Grade each `(query, candidate)` pair on the BEIR-standard 0–3
scale:

- **3** — highly relevant: directly answers the query.
- **2** — relevant: clearly addresses the query topic.
- **1** — marginally relevant: tangentially related.
- **0** — not relevant.

We force JSON output so we can parse it deterministically.

In [6]:
JUDGE_PROMPT = (
    "You are an information retrieval relevance judge. Given a search "
    "query and a passage from a document, grade how relevant the passage "
    "is to the query on this scale:\n"
    "  3 - Highly relevant: directly answers the query.\n"
    "  2 - Relevant: clearly addresses the query topic.\n"
    "  1 - Marginally relevant: tangentially related.\n"
    "  0 - Not relevant.\n\n"
    "Query  : {query}\n"
    "Passage: {passage}\n\n"
    "Output ONLY a JSON object: "
    '{{"grade": <0|1|2|3>, "reason": "<short>"}}'
)

def judge(query: str, passage: str) -> tuple[int, str]:
    resp = oai.chat.completions.create(
        model           = LLM_MODEL,
        temperature     = 0.0,
        response_format = {"type": "json_object"},
        messages        = [{"role": "user",
                            "content": JUDGE_PROMPT.format(
                                query=query, passage=passage[:1500])}],
    )
    data = json.loads(resp.choices[0].message.content)
    return int(data.get('grade', 0)), data.get('reason', '')

draft_qrels = {}
reasons     = {}
done = 0
for q in draft_queries:
    qid = q['qid']
    draft_qrels[qid] = {}
    for row in pool[qid]:
        grade, reason = judge(q['query'], row['text'])
        draft_qrels[qid][row['doc_id']] = grade
        reasons[(qid, row['doc_id'])]   = reason
        done += 1
        if done % 10 == 0:
            print(f"  graded {done} / ~{sum(len(v) for v in pool.values())}")
print(f"  graded {done} pairs.")

  graded 10 / ~160


  graded 20 / ~160


  graded 30 / ~160


  graded 40 / ~160


  graded 50 / ~160


...

  graded 120 / ~160


  graded 130 / ~160


  graded 140 / ~160


  graded 150 / ~160


  graded 160 / ~160
  graded 160 pairs.


## Step 5 — Review and curate

This is the critical human-in-the-loop step. The LLM judge is
a *draft*. Skim the high-grade and low-grade ends — that's
where mistakes hide. Tweak any cell of `draft_qrels` directly.
A small, carefully reviewed set beats a large, blindly-trusted
one.

In [7]:
import pandas as pd

review_rows = []
for q in draft_queries:
    qid = q['qid']
    for row in pool[qid]:
        did = row['doc_id']
        review_rows.append({
            "qid"   : qid,
            "query" : q['query'][:50],
            "doc_id": did,
            "grade" : draft_qrels[qid].get(did, 0),
            "reason": reasons.get((qid, did), '')[:60],
            "title" : (row.get('title') or '')[:50],
        })
review_df = pd.DataFrame(review_rows).sort_values(
    ["qid", "grade"], ascending=[True, False],
)
review_df.head(20)

,qid,query,doc_id,grade,reason,title
0,my_q_001,NVP-BEZ235 effects on cancer cells with PI3K m...,14241418,3,The passage directly discusses the effects of ...,"NVP-BEZ235, a dual PI3K/mTOR inhibitor, preven..."
1,my_q_001,NVP-BEZ235 effects on cancer cells with PI3K m...,4961038,3,The passage directly discusses the effects of ...,Effective Use of PI3K and MEK Inhibitors to Tr...
2,my_q_001,NVP-BEZ235 effects on cancer cells with PI3K m...,1285713,1,"The passage discusses PI103, a compound relate...",Pharmacologic characterization of a potent inh...
3,my_q_001,NVP-BEZ235 effects on cancer cells with PI3K m...,33387953,1,The passage discusses the effects of BEZ235 in...,Mutations in G protein beta subunits promote t...
4,my_q_001,NVP-BEZ235 effects on cancer cells with PI3K m...,14819804,1,The passage discusses PI3K inhibitors and thei...,Mutations in the phosphatidylinositol-3-kinase...
5,my_q_001,NVP-BEZ235 effects on cancer cells with PI3K m...,24294572,1,The passage discusses the PI3K signaling pathw...,"PTEN Regulates PI(3,4)P2 Signaling Downstream ..."
6,my_q_001,NVP-BEZ235 effects on cancer cells with PI3K m...,18399038,1,The passage discusses glioma tumor-initiating ...,Establishment of human iPSC-based models for t...
7,my_q_001,NVP-BEZ235 effects on cancer cells with PI3K m...,4414547,0,The passage discusses genetic variations relat...,Mosaic PPM1D mutations are associated with pre...
8,my_q_002,how mesenchymal inflammation affects hematopoi...,3863543,3,The passage directly addresses how mesenchymal...,Mesenchymal Inflammation Drives Genotoxic Stre...
9,my_q_002,how mesenchymal inflammation affects hematopoi...,5107861,1,The passage mentions hematopoietic stem cells ...,Chronic variable stress activates hematopoieti...


In [8]:
# Want to override a judgement? Edit it directly. Example:
#   draft_qrels["my_q_001"]["12345"] = 2
# then re-run the previous cell to see the updated table.

## Step 6 — Save and re-evaluate

Write the curated qrels to disk, then run the three search
strategies from Lab 2 against *your* qrels. These numbers
tell you which strategy works best **on your data**, not on
BEIR's.

In [9]:
EVAL_DIR = pathlib.Path(_REPO_ROOT) / "eval_sets"
EVAL_DIR.mkdir(exist_ok=True)

(EVAL_DIR / f"{DATASET}_custom_queries.json").write_text(
    json.dumps({q['qid']: q['query'] for q in draft_queries}, indent=2),
)
(EVAL_DIR / f"{DATASET}_custom_qrels.json").write_text(
    json.dumps(draft_qrels, indent=2),
)
print(f"Wrote {EVAL_DIR}")
print(f"  {DATASET}_custom_queries.json")
print(f"  {DATASET}_custom_qrels.json")

Wrote /Users/henryweller/Downloads/voyage-context-3-testing/eval_sets
  scifact_custom_queries.json
  scifact_custom_qrels.json


In [10]:
def query_metrics(ranked, qrels_for_q, ks=(5, 10)):
    rel_set = {did for did, s in qrels_for_q.items() if s > 0}
    out = {}
    for k in ks:
        top_k = ranked[:k]
        out[f"P@{k}"] = sum(1 for d in top_k if d in rel_set) / k
        out[f"R@{k}"] = len(set(top_k) & rel_set) / max(1, len(rel_set))
        dcg = sum(qrels_for_q.get(d, 0) / math.log2(i + 1)
                  for i, d in enumerate(top_k, 1))
        ideal = sorted((s for s in qrels_for_q.values() if s > 0), reverse=True)[:k]
        idcg = sum(g / math.log2(i + 1) for i, g in enumerate(ideal, 1))
        out[f"NDCG@{k}"] = dcg / idcg if idcg else 0.0
    out["MRR"] = next(
        (1.0 / i for i, d in enumerate(ranked, 1) if d in rel_set), 0.0,
    )
    cum = hits = 0
    for i, d in enumerate(ranked, 1):
        if d in rel_set:
            hits += 1
            cum  += hits / i
    out["AP"] = cum / len(rel_set) if rel_set else 0.0
    return out


def dedup(rows, k):
    seen = {}
    for r in rows:
        did = r['doc_id']
        if did not in seen or r['score'] > seen[did]['score']:
            seen[did] = r
    return sorted(seen.values(), key=lambda r: r['score'], reverse=True)[:k]

def bm25_search(q_text, top_k=TOP_K):
    pipeline = [
        {"$search": {"index": TEXT_INDEX_NAME,
                      "text": {"path": "text", "query": q_text}}},
        {"$addFields": {"score": {"$meta": "searchScore"}}},
        {"$limit": top_k * 4},
        {"$sort":  {"score": -1}},
    ]
    return [r['doc_id'] for r in dedup(coll.aggregate(pipeline), top_k)]

def vector_search(q_vec, top_k=TOP_K):
    pipeline = [
        {"$vectorSearch": {
            "index": VECTOR_INDEX_NAME, "path": "embedding",
            "queryVector": q_vec,
            "numCandidates": top_k * 20, "limit": top_k * 4,
        }},
        {"$addFields": {"score": {"$meta": "vectorSearchScore"}}},
        {"$sort": {"score": -1}},
    ]
    return [r['doc_id'] for r in dedup(coll.aggregate(pipeline), top_k)]

results = {}
for name, search in [
    ("lexical (BM25)", lambda v, t: bm25_search(t)),
    ("vector",         lambda v, t: vector_search(v)),
    ("hybrid α=0.8",   lambda v, t: [r['doc_id'] for r in hybrid_search(v, t, alpha=0.8)]),
]:
    per_q = []
    for q, qv in zip(draft_queries, q_vecs):
        ranked = search(qv, q['query'])
        per_q.append(query_metrics(ranked, draft_qrels[q['qid']]))
    keys = per_q[0].keys()
    agg = {k: sum(p[k] for p in per_q) / len(per_q) for k in keys}
    agg["MAP"] = agg.pop("AP")
    results[name] = agg

pd.DataFrame(results).T[["P@5", "R@5", "NDCG@5", "NDCG@10", "MRR", "MAP"]].round(3)

,P@5,R@5,NDCG@5,NDCG@10,MRR,MAP
lexical (BM25),0.51,0.565,0.815,0.801,1.0,0.563
vector,0.67,0.751,0.917,0.930,1.0,0.819
hybrid α=0.8,0.71,0.791,0.936,0.973,1.0,0.916


## Where to go next

You've just built a tiny custom evaluation set. To take it
further:

- **Scale up.** 20 queries is a smoke test; aim for 100+
  before trusting the numbers to drive decisions.
- **Diversify queries.** Sample from real user logs if you
  have them. If not, prompt the LLM for multiple *styles*
  (keyword, natural-language, multi-hop).
- **Tighten the judging prompt.** Add a couple of in-prompt
  examples from your domain. Calibrate the LLM on a handful
  of human-graded examples before scaling.
- **Re-curate.** Eval sets decay as corpora and user needs
  change. Treat them as living code, not finished artefacts.

Once you have a trustworthy eval set, the **advanced** lab
under `phase4/` is where you'd go to push retrieval quality
further — per-query strategy routing, query rewriters
(HyDE, decompose), cross-encoder reranking. None of those
are worth tuning without an eval set you trust. That's
the whole point.